# 계층적 인덱싱

지금까지 우리는 주로 Pandas 'Series' 및 'DataFrame' 개체에 저장된 1차원 및 2차원 데이터에 중점을 두었습니다.
종종 이를 뛰어넘어 고차원 데이터, 즉 하나 또는 두 개 이상의 키로 인덱싱된 데이터를 저장하는 것이 유용합니다.
초기 Pandas 버전은 2D `DataFrame`에 대한 3D 또는 4D 아날로그로 생각할 수 있는 `Panel` 및 `Panel4D` 개체를 제공했지만 실제로 사용하기에는 다소 투박했습니다. 고차원 데이터를 처리하는 훨씬 더 일반적인 패턴은 *계층적 인덱싱*(*다중 인덱싱*이라고도 함)을 사용하여 단일 인덱스 내에 여러 인덱스 *수준*을 통합하는 것입니다.
이러한 방식으로 고차원 데이터는 익숙한 1차원 `Series` 객체와 2차원 `DataFrame` 객체 내에서 간결하게 표현될 수 있습니다.
(Pandas 스타일의 유연한 인덱스가 있는 진정한 *N* 차원 배열에 관심이 있다면 뛰어난 [Xarray 패키지](https://xarray.pydata.org/)를 살펴보세요.)

이번 장에서는 `MultiIndex` 객체를 직접 생성하는 방법을 살펴보겠습니다. 다중 인덱싱된 데이터에 대한 인덱싱, 슬라이싱 및 통계 계산 시 고려 사항 간단한 데이터 표현과 계층적으로 인덱스된 데이터 표현 사이를 변환하는 데 유용한 루틴입니다.

표준 가져오기부터 시작합니다.

In [1]:
import pandas as pd
import numpy as np

## 곱셈 인덱스 시리즈

1차원 `시리즈` 내에서 2차원 데이터를 어떻게 표현할 수 있는지 고려하는 것부터 시작해 보겠습니다.
구체적으로 설명하기 위해 각 지점에 문자와 숫자 키가 있는 일련의 데이터를 고려해 보겠습니다.

### 나쁜 방법

서로 다른 두 해의 주에 대한 데이터를 추적하고 싶다고 가정해 보겠습니다.
이미 다룬 Pandas 도구를 사용하면 단순히 파이썬(Python) 튜플을 키로 사용하고 싶은 유혹을 느낄 수도 있습니다.

In [2]:
index = [('California', 2010), ('California', 2020),
         ('New York', 2010), ('New York', 2020),
         ('Texas', 2010), ('Texas', 2020)]
populations = [37253956, 39538223,
               19378102, 20201249,
               25145561, 29145505]
pop = pd.Series(populations, index=index)
pop

(California, 2010)    37253956
(California, 2020)    39538223
(New York, 2010)      19378102
(New York, 2020)      20201249
(Texas, 2010)         25145561
(Texas, 2020)         29145505
dtype: int64

이 인덱싱 방식을 사용하면 이 튜플 인덱스를 기반으로 계열을 직접 인덱싱하거나 분할할 수 있습니다.

In [3]:
pop[('California', 2020):('Texas', 2010)]

(California, 2020)    39538223
(New York, 2010)      19378102
(New York, 2020)      20201249
(Texas, 2010)         25145561
dtype: int64

그러나 편리함은 거기서 끝입니다. 예를 들어 2010년의 모든 값을 선택해야 하는 경우 이를 실현하기 위해 지저분하고 잠재적으로 느린 작업을 수행해야 합니다.

In [4]:
pop[[i for i in pop.index if i[1] == 2010]]

(California, 2010)    37253956
(New York, 2010)      19378102
(Texas, 2010)         25145561
dtype: int64

이는 원하는 결과를 생성하지만 Pandas에서 우리가 좋아하게 된 슬라이싱 구문만큼 깨끗하지 않습니다(또는 대규모 데이터 세트의 경우 효율적이지 않습니다).

### 더 나은 방법: Pandas MultiIndex
다행히 Pandas는 더 나은 방법을 제공합니다.
튜플 기반 인덱싱은 본질적으로 기본적인 다중 인덱스이며 Pandas 'MultiIndex' 유형은 우리가 원하는 작업 유형을 제공합니다.
다음과 같이 튜플에서 다중 인덱스를 만들 수 있습니다.

In [5]:
index = pd.MultiIndex.from_tuples(index)

'MultiIndex'는 여러 색인 *레벨*(이 경우에는 주 이름과 연도)뿐만 아니라 이러한 수준을 인코딩하는 각 데이터 포인트에 대한 여러 *레이블*을 나타냅니다.

이 `MultiIndex`를 사용하여 시리즈를 다시 색인화하면 데이터의 계층적 표현을 살펴볼 수 있습니다.

In [6]:
pop = pop.reindex(index)
pop

California  2010    37253956
            2020    39538223
New York    2010    19378102
            2020    20201249
Texas       2010    25145561
            2020    29145505
dtype: int64

여기서 시리즈 표현의 처음 두 열은 여러 인덱스 값을 표시하고 세 번째 열은 데이터를 표시합니다.
첫 번째 열에는 일부 항목이 누락되어 있습니다. 이 다중 인덱스 표현에서 빈 항목은 그 위의 줄과 동일한 값을 나타냅니다.

이제 두 번째 인덱스가 2020인 모든 데이터에 액세스하려면 Pandas 슬라이싱 표기법을 사용할 수 있습니다.

In [7]:
pop[:, 2020]

California    39538223
New York      20201249
Texas         29145505
dtype: int64

결과는 우리가 관심 있는 키만 포함하는 단일 인덱스 시리즈입니다.
이 구문은 우리가 시작한 홈 스펀 튜플 기반 다중 인덱싱 솔루션보다 훨씬 더 편리합니다(그리고 작업이 훨씬 더 효율적입니다!).
이제 계층적으로 인덱싱된 데이터에 대한 이러한 종류의 인덱싱 작업에 대해 자세히 설명하겠습니다.

### 추가 차원으로서의 MultiIndex

여기서 다른 점을 발견할 수 있습니다. 인덱스 및 열 레이블이 있는 간단한 'DataFrame'을 사용하여 동일한 데이터를 쉽게 저장할 수 있었습니다.
실제로 Pandas는 이러한 동등성을 염두에 두고 구축되었습니다. `unstack` 메서드는 다중 색인 `Series`를 기존 색인 `DataFrame`으로 빠르게 변환합니다.

In [8]:
pop_df = pop.unstack()
pop_df

,2010,2020
California,37253956,39538223
New York,19378102,20201249
Texas,25145561,29145505


당연히 ``stack`` 방법은 반대 작업을 제공합니다.

In [9]:
pop_df.stack()

California  2010    37253956
            2020    39538223
New York    2010    19378102
            2020    20201249
Texas       2010    25145561
            2020    29145505
dtype: int64

이것을 보면 왜 우리가 계층적 인덱싱에 신경을 쓰려고 하는지 궁금할 것입니다.
그 이유는 간단합니다. 다중 인덱싱을 사용하여 1차원 `시리즈` 내에서 2차원 데이터를 조작할 수 있었던 것과 마찬가지로 이를 사용하여 `시리즈` 또는 `DataFrame`에서 3차원 이상의 데이터를 조작할 수도 있습니다.
다중 인덱스의 각 추가 수준은 데이터의 추가 차원을 나타냅니다. 이 속성을 활용하면 표현할 수 있는 데이터 유형에 훨씬 더 많은 유연성이 제공됩니다. 구체적으로, 우리는 매년 각 주에 대한 또 다른 인구통계 데이터 열을 추가할 수 있습니다(예: 18세 미만 인구). 'MultiIndex'를 사용하면 ``DataFrame``에 다른 열을 추가하는 것만큼 쉽습니다.

In [10]:
pop_df = pd.DataFrame({'total': pop,
                       'under18': [9284094, 8898092,
                                   4318033, 4181528,
                                   6879014, 7432474]})
pop_df

total  under18
California 2010  37253956  9284094
           2020  39538223  8898092
New York   2010  19378102  4318033
           2020  20201249  4181528
Texas      2010  25145561  6879014
           2020  29145505  7432474

또한 [Pandas의 데이터 작업](03.03-Operations-in-Pandas.ipynb)에서 설명한 모든 ufunc 및 기타 기능은 계층적 인덱스에서도 작동합니다.
여기서는 위 데이터를 바탕으로 18세 미만의 인구 비율을 연도별로 계산합니다.

In [11]:
f_u18 = pop_df['under18'] / pop_df['total']
f_u18.unstack()

,2010,2020
California,0.249211,0.225050
New York,0.222831,0.206994
Texas,0.273568,0.255013


이를 통해 고차원 데이터도 쉽고 빠르게 조작하고 탐색할 수 있습니다.

## MultiIndex 생성 방법

다중 인덱스 `Series` 또는 `DataFrame`을 구성하는 가장 간단한 방법은 두 개 이상의 인덱스 배열 목록을 생성자에 전달하는 것입니다. 예를 들어:

In [12]:
df = pd.DataFrame(np.random.rand(4, 2),
                  index=[['a', 'a', 'b', 'b'], [1, 2, 1, 2]],
                  columns=['data1', 'data2'])
df

data1     data2
a 1  0.748464  0.561409
  2  0.379199  0.622461
b 1  0.701679  0.687932
  2  0.436200  0.950664

``MultiIndex`` 생성 작업은 백그라운드에서 수행됩니다.

마찬가지로 적절한 튜플을 키로 사용하여 사전을 전달하면 Pandas는 이를 자동으로 인식하고 기본적으로 ``MultiIndex``를 사용합니다.

In [13]:
data = {('California', 2010): 37253956,
        ('California', 2020): 39538223,
        ('New York', 2010): 19378102,
        ('New York', 2020): 20201249,
        ('Texas', 2010): 25145561,
        ('Texas', 2020): 29145505}
pd.Series(data)

California  2010    37253956
            2020    39538223
New York    2010    19378102
            2020    20201249
Texas       2010    25145561
            2020    29145505
dtype: int64

그럼에도 불구하고 때로는 'MultiIndex'를 명시적으로 생성하는 것이 유용할 수 있습니다. 다음에는 이 작업을 수행하는 몇 가지 방법을 살펴보겠습니다.

### 명시적 MultiIndex 생성자

인덱스 구성 방법에 대한 유연성을 높이려면 `pd.MultiIndex` 클래스에서 제공되는 생성자 메서드를 대신 사용할 수 있습니다.
예를 들어 이전에 했던 것처럼 각 레벨 내의 인덱스 값을 제공하는 간단한 배열 목록에서 `MultiIndex`를 구성할 수 있습니다.

In [14]:
pd.MultiIndex.from_arrays([['a', 'a', 'b', 'b'], [1, 2, 1, 2]])

MultiIndex([('a', 1),
            ('a', 2),
            ('b', 1),
            ('b', 2)],
           )

또는 각 지점의 여러 인덱스 값을 제공하는 튜플 목록에서 이를 구성할 수 있습니다.

In [15]:
pd.MultiIndex.from_tuples([('a', 1), ('a', 2), ('b', 1), ('b', 2)])

MultiIndex([('a', 1),
            ('a', 2),
            ('b', 1),
            ('b', 2)],
           )

단일 인덱스의 데카르트 곱으로 구성할 수도 있습니다.

In [16]:
pd.MultiIndex.from_product([['a', 'b'], [1, 2]])

MultiIndex([('a', 1),
            ('a', 2),
            ('b', 1),
            ('b', 2)],
           )

마찬가지로 `levels`(각 레벨에 사용 가능한 인덱스 값이 포함된 목록 목록) 및 `codes`(이러한 라벨을 참조하는 목록 목록)를 전달하여 내부 인코딩을 사용하여 직접 `MultiIndex`를 구성할 수 있습니다.

In [17]:
pd.MultiIndex(levels=[['a', 'b'], [1, 2]],
              codes=[[0, 0, 1, 1], [0, 1, 0, 1]])

MultiIndex([('a', 1),
            ('a', 2),
            ('b', 1),
            ('b', 2)],
           )

이러한 개체는 `Series` 또는 `DataFrame`을 생성할 때 `index` 인수로 전달되거나 기존 `Series` 또는 `DataFrame`의 `reindex` 메서드에 전달될 수 있습니다.

### MultiIndex 수준 이름

때로는 `MultiIndex`의 수준에 이름을 지정하는 것이 편리합니다.
이는 앞에서 설명한 `MultiIndex` 생성자에 `names` 인수를 전달하거나 사실 뒤에 인덱스의 `names` 속성을 설정하여 수행할 수 있습니다.

In [18]:
pop.index.names = ['state', 'year']
pop

state       year
California  2010    37253956
            2020    39538223
New York    2010    19378102
            2020    20201249
Texas       2010    25145561
            2020    29145505
dtype: int64

더 많은 데이터 세트가 포함된 경우 이는 다양한 인덱스 값의 의미를 추적하는 유용한 방법이 될 수 있습니다.

### 열용 MultiIndex

`DataFrame`에서 행과 열은 완전히 대칭이며 행이 여러 수준의 인덱스를 가질 수 있는 것처럼 열도 여러 수준을 가질 수 있습니다.
일부 (다소 현실적인) 의료 데이터의 모형인 다음을 고려하십시오.

In [19]:
# hierarchical indices and columns
index = pd.MultiIndex.from_product([[2013, 2014], [1, 2]],
                                   names=['year', 'visit'])
columns = pd.MultiIndex.from_product([['Bob', 'Guido', 'Sue'], ['HR', 'Temp']],
                                     names=['subject', 'type'])

# mock some data
data = np.round(np.random.randn(4, 6), 1)
data[:, ::2] *= 10
data += 37

# create the DataFrame
health_data = pd.DataFrame(data, index=index, columns=columns)
health_data

subject      Bob       Guido         Sue      
type          HR  Temp    HR  Temp    HR  Temp
year visit                                    
2013 1      30.0  38.0  56.0  38.3  45.0  35.8
     2      47.0  37.1  27.0  36.0  37.0  36.4
2014 1      51.0  35.9  24.0  36.7  32.0  36.2
     2      49.0  36.3  48.0  39.2  31.0  35.7

이는 기본적으로 대상, 측정 유형, 연도, 방문 횟수를 차원으로 하는 4차원 데이터입니다.
예를 들어 이를 사용하면 사람 이름으로 최상위 열을 색인화하고 해당 사람의 정보만 포함하는 전체 'DataFrame'을 얻을 수 있습니다.

In [20]:
health_data['Guido']

type          HR  Temp
year visit            
2013 1      56.0  38.3
     2      27.0  36.0
2014 1      24.0  36.7
     2      48.0  39.2

## MultiIndex 인덱싱 및 슬라이싱

'MultiIndex'의 인덱싱 및 슬라이싱은 직관적으로 설계되었으며 인덱스를 추가된 차원으로 생각하면 도움이 됩니다.
먼저 인덱싱된 `Series`를 인덱싱하고 곱한 다음 인덱싱된 `DataFrame` 객체를 곱하는 방법을 살펴보겠습니다.

### 인덱스 시리즈 곱하기

앞서 본 주 인구의 다중 색인 '계열'을 고려해보세요.

In [21]:
pop

state       year
California  2010    37253956
            2020    39538223
New York    2010    19378102
            2020    20201249
Texas       2010    25145561
            2020    29145505
dtype: int64

여러 용어로 색인을 생성하여 단일 요소에 액세스할 수 있습니다.

In [22]:
pop['California', 2010]

37253956

'MultiIndex'는 *부분 인덱싱*, 즉 인덱스 수준 중 하나만 인덱싱하는 기능도 지원합니다.
결과는 하위 수준 인덱스가 유지되는 또 다른 '시리즈'입니다.

In [23]:
pop['California']

year
2010    37253956
2020    39538223
dtype: int64

'MultiIndex'가 정렬되어 있는 한 부분 슬라이싱도 가능합니다([정렬 및 정렬되지 않은 인덱스](#Sorted-and-unsorted-indices)의 설명 참조).

In [24]:
pop.loc['California':'New York']

state       year
California  2010    37253956
            2020    39538223
New York    2010    19378102
            2020    20201249
dtype: int64

정렬된 인덱스를 사용하면 첫 번째 인덱스에 빈 조각을 전달하여 하위 수준에서 부분 인덱싱을 수행할 수 있습니다.

In [25]:
pop[:, 2010]

state
California    37253956
New York      19378102
Texas         25145561
dtype: int64

다른 유형의 인덱싱 및 선택([데이터 인덱싱 및 선택](03.02-Data-Indexing-and-Selection.ipynb)에서 설명)도 작동합니다. 예를 들어 부울 마스크를 기반으로 한 선택:

In [26]:
pop[pop > 22000000]

state       year
California  2010    37253956
            2020    39538223
Texas       2010    25145561
            2020    29145505
dtype: int64

팬시 인덱싱을 기반으로 한 선택도 가능합니다.

In [27]:
pop[['California', 'Texas']]

state       year
California  2010    37253956
            2020    39538223
Texas       2010    25145561
            2020    29145505
dtype: int64

### 인덱싱된 DataFrame 곱하기

다중 인덱스 `DataFrame`은 비슷한 방식으로 작동합니다.
이전의 장난감 의료 `DataFrame`을 생각해 보세요.

In [28]:
health_data

subject      Bob       Guido         Sue      
type          HR  Temp    HR  Temp    HR  Temp
year visit                                    
2013 1      30.0  38.0  56.0  38.3  45.0  35.8
     2      47.0  37.1  27.0  36.0  37.0  36.4
2014 1      51.0  35.9  24.0  36.7  32.0  36.2
     2      49.0  36.3  48.0  39.2  31.0  35.7

열은 `DataFrame`의 기본 열이며 다중 색인 `Series`에 사용되는 구문이 열에 적용된다는 점을 기억하세요.
예를 들어 간단한 작업으로 Guido의 심박수 데이터를 복구할 수 있습니다.

In [29]:
health_data['Guido', 'HR']

year  visit
2013  1        56.0
      2        27.0
2014  1        24.0
      2        48.0
Name: (Guido, HR), dtype: float64

또한 단일 인덱스의 경우와 마찬가지로 [데이터 인덱싱 및 선택](03.02-Data-Indexing-and-Selection.ipynb)에 소개된 `loc`, `iloc`, `ix` 인덱서를 사용할 수 있습니다. 예를 들어:

In [30]:
health_data.iloc[:2, :2]

subject      Bob      
type          HR  Temp
year visit            
2013 1      30.0  38.0
     2      47.0  37.1

이러한 인덱서는 기본 2차원 데이터에 대한 배열과 유사한 보기를 제공하지만 'loc' 또는 'iloc'의 각 개별 인덱스는 여러 인덱스의 튜플을 전달할 수 있습니다. 예를 들어:

In [31]:
health_data.loc[:, ('Bob', 'HR')]

year  visit
2013  1        30.0
      2        47.0
2014  1        51.0
      2        49.0
Name: (Bob, HR), dtype: float64

이러한 인덱스 튜플 내에서 슬라이스로 작업하는 것은 특별히 편리하지 않습니다. 튜플 내에 슬라이스를 만들려고 하면 구문 오류가 발생합니다.

In [32]:
health_data.loc[(:, 1), (:, 'HR')]

SyntaxError: invalid syntax (3311942670.py, line 1)

파이썬(Python)에 내장된 `slice` 함수를 사용하여 원하는 슬라이스를 명시적으로 구축하여 이 문제를 해결할 수 있지만, 이 맥락에서 더 좋은 방법은 Pandas가 정확하게 이 상황에 제공하는 `IndexSlice` 객체를 사용하는 것입니다.
예를 들어:

In [33]:
idx = pd.IndexSlice
health_data.loc[idx[:, 1], idx[:, 'HR']]

,subject,Bob,Guido,Sue
,type,HR,HR,HR
year,visit,,,
2013,1,30.0,56.0,45.0
2014,1,51.0,24.0,32.0


보시다시피, 다중 인덱스 `Series` 및 `DataFrame`에서 데이터와 상호 작용하는 방법은 여러 가지가 있으며, 이 책에 있는 많은 도구와 마찬가지로 이러한 도구에 익숙해지는 가장 좋은 방법은 직접 사용해 보는 것입니다!

## 다중 인덱스 재배열

다중 인덱스 데이터 작업의 핵심 중 하나는 데이터를 효과적으로 변환하는 방법을 아는 것입니다.
데이터 세트의 모든 정보를 보존하지만 다양한 계산 목적에 맞게 재배열하는 여러 작업이 있습니다.
이에 대한 간략한 예는 `stack` 및 `unstack` 방법에서 보았지만 계층적 인덱스와 열 간의 데이터 재배열을 미세하게 제어할 수 있는 더 많은 방법이 있으므로 여기서 살펴보겠습니다.

### 정렬된 인덱스와 정렬되지 않은 인덱스

앞서 간략하게 주의사항을 언급했지만 여기서는 더 강조하고 싶습니다.
*인덱스가 정렬되지 않으면 'MultiIndex' 슬라이싱 작업 중 다수가 실패합니다.*
좀 더 자세히 살펴보겠습니다.

인덱스가 *사전순으로 정렬되지 않은* 간단한 다중 인덱스 데이터를 만드는 것부터 시작하겠습니다.

In [34]:
index = pd.MultiIndex.from_product([['a', 'c', 'b'], [1, 2]])
data = pd.Series(np.random.rand(6), index=index)
data.index.names = ['char', 'int']
data

char  int
a     1      0.280341
      2      0.097290
c     1      0.206217
      2      0.431771
b     1      0.100183
      2      0.015851
dtype: float64

이 인덱스의 일부 조각을 가져오려고 하면 오류가 발생합니다.

In [35]:
try:
    data['a':'b']
except KeyError as e:
    print("KeyError", e)

KeyError 'Key length (1) was greater than MultiIndex lexsort depth (0)'


오류 메시지에서 완전히 명확하지는 않지만 `MultiIndex`가 정렬되지 않은 결과입니다.
여러 가지 이유로 부분 슬라이스 및 기타 유사한 작업을 수행하려면 'MultiIndex'의 수준이 정렬된(즉, 사전순) 순서로 되어 있어야 합니다.
Pandas는 `DataFrame`의 `sort_index` 및 `sortlevel` 메서드와 같이 이러한 유형의 정렬을 수행하기 위한 다양한 편의 루틴을 제공합니다.
여기서는 가장 간단한 `sort_index`를 사용하겠습니다.

In [36]:
data = data.sort_index()
data

char  int
a     1      0.280341
      2      0.097290
b     1      0.100183
      2      0.015851
c     1      0.206217
      2      0.431771
dtype: float64

이런 방식으로 인덱스를 정렬하면 부분 슬라이싱이 예상대로 작동합니다.

In [37]:
data['a':'b']

char  int
a     1      0.280341
      2      0.097290
b     1      0.100183
      2      0.015851
dtype: float64

### 인덱스 스태킹 및 언스택

이전에 간략하게 살펴보았듯이, 누적된 다중 인덱스에서 간단한 2차원 표현으로 데이터세트를 변환하고 선택적으로 사용할 수준을 지정하는 것이 가능합니다.

In [38]:
pop.unstack(level=0)

state,California,New York,Texas
year,,,
2010,37253956,19378102,25145561
2020,39538223,20201249,29145505


In [39]:
pop.unstack(level=1)

year,2010,2020
state,,
California,37253956,39538223
New York,19378102,20201249
Texas,25145561,29145505


'unstack'의 반대는 'stack'입니다. 여기서는 원래 시리즈를 복구하는 데 사용할 수 있습니다.

In [40]:
pop.unstack().stack()

state       year
California  2010    37253956
            2020    39538223
New York    2010    19378102
            2020    20201249
Texas       2010    25145561
            2020    29145505
dtype: int64

### 인덱스 설정 및 재설정

계층적 데이터를 재배열하는 또 다른 방법은 인덱스 레이블을 열로 바꾸는 것입니다. 이는 `reset_index` 메소드를 사용하여 수행할 수 있습니다.
인구 사전에서 이를 호출하면 이전에 인덱스에 있었던 정보를 보유하는 `state` 및 `year` 열이 있는 `DataFrame`이 생성됩니다.
명확성을 위해 선택적으로 열 표현에 대한 데이터 이름을 지정할 수 있습니다.

In [41]:
pop_flat = pop.reset_index(name='population')
pop_flat

,state,year,population
0,California,2010,37253956
1,California,2020,39538223
2,New York,2010,19378102
3,New York,2020,20201249
4,Texas,2010,25145561
5,Texas,2020,29145505


일반적인 패턴은 열 값에서 `MultiIndex`를 작성하는 것입니다.
이는 다중 인덱스 `DataFrame`을 반환하는 `DataFrame`의 `set_index` 메서드를 사용하여 수행할 수 있습니다.

In [42]:
pop_flat.set_index(['state', 'year'])

population
state      year            
California 2010    37253956
           2020    39538223
New York   2010    19378102
           2020    20201249
Texas      2010    25145561
           2020    29145505

실제로 이러한 유형의 재인덱싱은 실제 데이터 세트를 탐색할 때 더 유용한 패턴 중 하나입니다.